# Cost-оптимальный порог и операционная точка — Elliptic++

Квантиль скора ("алертим топ-10%") выглядит удобно, но он **циркулярен**: порог выбирается из распределения скора, а не из бизнес-стоимости ошибок. В этом ноутбуке переходим от квантилей к **cost-оптимальному** порогу $\tau^{\ast}=\arg\min_\tau C_{FP}FP(\tau)+C_{FN}FN(\tau)$ и смотрим, как бюджет аналитиков ограничивает эту оптимизацию.

**План:**
- Temporal split 1..30 / 31..40 / 41..49 — обучение RF-прокси на train, все пороги калибруем на **valid**.
- PR-кривая на valid → PR-AUC vs baseline.
- Cost-кривые для $C_{FN}=10$ и $100$ ($C_{FP}=1$) → $\tau^{\ast}$ + бюджет $B=500$ алертов.
- Tier 1/2/3 → precision/recall/FTE и сравнение с квантилем 90%.
- Power analysis для оценки precision фазы auto-clear.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT (notebook runs from docs/notebooks or repo root)
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Elliptic data not found, tried: {candidates}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")


## 1. Загрузка и обучение скора

Читаем только через `load_elliptic`, фильтруем `labeled` (`class ∈ {1,2}`), $y = 1$ iff `class==1` (illicit). Temporal split фиксирован: train 1..30, valid 31..40, test 41..49. На train обучаем `RandomForest(n=200, class_weight=balanced)` — прокси для GBDT. Все пороги калибруем на **valid**, test не трогаем до конца.


In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features {features.shape}  classes {classes.shape}  edgelist {edgelist.shape}  merged {merged.shape}")
print(f"time_step range: {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged["class"].value_counts(dropna=False).to_frame("n"))

# filter labeled only
df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"labeled: {len(df):,}  illicit {df['y'].mean():.2%}  feat {len(feat_cols)}")

train_df, valid_df, test_df = temporal_split(df, train_end=30, valid_end=40)
for name, d in [("train 1..30", train_df), ("valid 31..40", valid_df), ("test  41..49", test_df)]:
    print(f"{name}: n={len(d):,}  illicit={d['y'].mean():.4f}  time {d['time_step'].min()}..{d['time_step'].max()}")

X_train = train_df[feat_cols].values
X_valid = valid_df[feat_cols].values
X_test = test_df[feat_cols].values
y_train, y_valid, y_test = train_df["y"].values, valid_df["y"].values, test_df["y"].values

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=72)
rf.fit(X_train, y_train)
print("RF обучен (200 trees, balanced)")

valid_score = rf.predict_proba(X_valid)[:, 1]
test_score = rf.predict_proba(X_test)[:, 1]
print(f"valid_score mean={valid_score.mean():.4f}  max={valid_score.max():.4f}")
print(f"test_score  mean={test_score.mean():.4f}  max={test_score.max():.4f}")


## 2. PR-кривая на valid — почему квантиль порочен

На дисбалансе (illicit ~10% среди labeled в train/valid, ~5% в test) accuracy бессмысленна. Смотрим на **Precision-Recall** и PR-AUC (`average_precision`). Baseline — доля позитивов (random ranker). Квантиль (например, 90-й перцентиль скора) **циркулярен**: он калибрует порог по распределению скора модели, а не по стоимости $FP/FN$, и плывёт при дрифте скора между периодами. Cost-подход фиксирует это — порог из $\min Cost$, а не из $Quantile(score)$.


In [ ]:
prec_v, rec_v, thr_v = precision_recall_curve(y_valid, valid_score)
ap_valid = average_precision_score(y_valid, valid_score)
baseline_valid = y_valid.mean()

prec_t, rec_t, thr_t = precision_recall_curve(y_test, test_score)
ap_test = average_precision_score(y_test, test_score)
baseline_test = y_test.mean()

print(f"valid: PR-AUC={ap_valid:.4f}  baseline={baseline_valid:.4f}  lift={ap_valid/baseline_valid:.1f}x")
print(f"test:  PR-AUC={ap_test:.4f}  baseline={baseline_test:.4f}  lift={ap_test/baseline_test:.1f}x")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# PR curve valid + test with baseline
axes[0].plot(rec_v, prec_v, label=f"valid PR-AUC={ap_valid:.3f}")
axes[0].plot(rec_t, prec_t, label=f"test PR-AUC={ap_test:.3f}", linestyle="--")
axes[0].axhline(baseline_valid, color="grey", linestyle=":", label=f"baseline valid={baseline_valid:.3f}")
axes[0].axhline(baseline_test, color="grey", linestyle="--", alpha=0.6, label=f"baseline test={baseline_test:.3f}")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("PR-кривая (valid — калибровка, test — дрифт)")
axes[0].legend(fontsize=8)

# score distribution by class (valid)
plot_df = pd.DataFrame({"score": valid_score, "y": y_valid})
plot_df["label"] = plot_df["y"].map({1: "illicit", 0: "licit"})
sns.histplot(data=plot_df, x="score", hue="label", bins=50, element="step", kde=True, ax=axes[1], alpha=0.5)
axes[1].set_title("Распределение скора на valid")
axes[1].set_xlabel("RF score")
plt.tight_layout()
plt.show()

# why quantile is circular — fixed percentile ignores cost and drifts
q90_valid = np.quantile(valid_score, 0.90)
q90_test = np.quantile(test_score, 0.90)
print(f"90-й перцентиль скора: valid={q90_valid:.3f}  test={q90_test:.3f}  -> порог плывёт с распределением")
print("Квантиль отвечает на вопрос 'сколько алертов хотим?', а не 'сколько стоят ошибки?' — поэтому при изменении C_FN/C_FP он не оптимален.")


## 3. Cost-функция и бюджет

$$Cost(\tau) = C_{FP}\cdot FP(\tau) + C_{FN}\cdot FN(\tau)$$

где $FP(\tau)$, $FN(\tau)$ — число ложных срабатываний и пропусков на **valid** при пороге $\tau$. Фиксируем $C_{FP}=1$ (час аналитика) и два сценария: $C_{FN}=10$ (умеренный риск) и $C_{FN}=100$ (высокий риск — пропуск отмывания дороже). Для каждого $\tau$ из `precision_recall_curve` считаем $FP/FN$ → $Cost$. Минимум даёт $\tau^{\ast}$. Отдельно смотрим **budget constraint**: при $C_{FN}/C_{FP}\gg1$ оптимум тянет $\tau\to0$ (алертить всё), но операционно $Alerts(\tau)\le B$, где $B=500$ — прокси дневного лимита очереди.


In [ ]:
C_FP = 1
scenarios = {"C_FN=10": 10, "C_FN=100": 100}

# thr_v from precision_recall_curve has len = len(prec)-1; align with prec/rec[:-1] is standard
# compute Cost for each threshold on valid
cost_df_list = []
for label, C_FN in scenarios.items():
    costs, fps, fns, alerts = [], [], [], []
    for t in thr_v:
        pred = (valid_score >= t).astype(int)
        fp = int(((pred == 1) & (y_valid == 0)).sum())
        fn = int(((pred == 0) & (y_valid == 1)).sum())
        costs.append(C_FP * fp + C_FN * fn)
        fps.append(fp)
        fns.append(fn)
        alerts.append(int((pred == 1).sum()))
    tmp = pd.DataFrame({"threshold": thr_v, "cost": costs, "FP": fps, "FN": fns, "alerts": alerts, "scenario": label, "C_FN": C_FN})
    cost_df_list.append(tmp)
cost_df = pd.concat(cost_df_list, ignore_index=True)

# find tau* per scenario
opt = cost_df.loc[cost_df.groupby("scenario")["cost"].idxmin()][["scenario", "threshold", "cost", "FP", "FN", "alerts"]]
opt = opt.rename(columns={"threshold": "tau_star"})
display(opt.to_string(index=False))
print(f"\nBUDGET B=500 алертов (прокси-дневной лимит на valid 31..40 = 10 периодов -> ~50/период, здесь — общий пул)")

tau_star_10 = float(opt.loc[opt["scenario"] == "C_FN=10", "tau_star"].iloc[0])
tau_star_100 = float(opt.loc[opt["scenario"] == "C_FN=100", "tau_star"].iloc[0])
print(f"tau* (C_FN=10)  = {tau_star_10:.4f}")
print(f"tau* (C_FN=100) = {tau_star_100:.4f}")


In [ ]:
B = 500  # budget: max alerts on valid (proxy for daily queue)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

# Cost vs tau — two scenarios
sns.lineplot(data=cost_df, x="threshold", y="cost", hue="scenario", ax=axes[0])
for _, row in opt.iterrows():
    axes[0].axvline(row["tau_star"], linestyle="--", alpha=0.7)
    axes[0].text(row["tau_star"], row["cost"], f"  {row['scenario']}\n  τ*={row['tau_star']:.3f}", va="bottom", fontsize=8)
axes[0].set_xlabel("τ (порог скора)")
axes[0].set_ylabel("Cost = C_FP·FP + C_FN·FN")
axes[0].set_title("Cost vs τ (valid) — два сценария")
axes[0].legend(title="")

# Alerts vs tau with budget line
sns.lineplot(data=cost_df, x="threshold", y="alerts", hue="scenario", ax=axes[1], legend=False)
axes[1].axhline(B, color="red", linestyle=":", label=f"budget B={B}")
axes[1].set_xlabel("τ")
axes[1].set_ylabel("Alerts(τ) на valid")
axes[1].set_title("Нагрузка на аналитиков: Alerts vs τ")
axes[1].legend()
# annotate feasible region
for _, row in opt.iterrows():
    feasible = "✓ feasible" if row["alerts"] <= B else "✗ over budget"
    axes[1].axvline(row["tau_star"], linestyle="--", alpha=0.6)
    axes[1].text(row["tau_star"], max(B, row["alerts"]), f" {feasible}", fontsize=7, va="bottom")

plt.tight_layout()
plt.show()

print("Интерпретация: при C_FN=100 τ* низкий → много алертов (3480 > B). Бюджет заставляет поднять τ до пересечения с B — иначе очередь захлебнётся, даже если формально Cost минимален.")
print("При C_FN=10 τ* высокий → алертов меньше, бюджет не биндит.")


## 4. Tier 1/2/3 и сравнение с квантилем

Разбиваем очередь на три уровня:
- **Tier 1** — $\tau_1 = \tau^{\ast}(C_{FN}=100)$ : максимальный риск, срочная ручная проверка.
- **Tier 2** — $\tau_2 = \tau^{\ast}(C_{FN}=10)$ : умеренный риск.
- **Tier 3** — $\tau_3 = 0.10$ : low-score хвост, bulk auto-clear кандидат.

Для каждого считаем precision/recall на **valid**, число алертов и ожидаемую нагрузку FTE = $Alerts \times t_{alert}$ (берём $t=0.25$ч ≈15 мин на алерт). Сравниваем с **квантильным baseline** — 90-й перцентиль скора (≈ топ-10% алертов).


In [ ]:
tau1 = tau_star_100
tau2 = tau_star_10
tau3 = 0.10
t_per_alert_h = 0.25  # 15 минут на алерт

def tier_metrics(y_true, scores, tau):
    pred = (scores >= tau).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    alerts = tp + fp
    fte_h = alerts * t_per_alert_h
    return {"tau": tau, "TP": tp, "FP": fp, "FN": fn, "TN": tn, "precision": prec, "recall": rec, "alerts": alerts, "FTE_h": fte_h}

tiers = {
    "Tier 1 (τ* C_FN=100) — urgent": tau1,
    "Tier 2 (τ* C_FN=10) — review": tau2,
    "Tier 3 (τ=0.10) — auto-clear cand.": tau3,
}

rows = []
for name, tau in tiers.items():
    m = tier_metrics(y_valid, valid_score, tau)
    m["tier"] = name
    rows.append(m)

# quantile baseline 90th percentile
q90 = float(np.quantile(valid_score, 0.90))
qm = tier_metrics(y_valid, valid_score, q90)
qm["tier"] = f"Quantile 90% (τ={q90:.3f}) — baseline"
rows.append(qm)

tier_df = pd.DataFrame(rows)[["tier", "tau", "alerts", "precision", "recall", "TP", "FP", "FN", "FTE_h"]]
tier_df["tau"] = tier_df["tau"].round(4)
tier_df["precision"] = tier_df["precision"].round(4)
tier_df["recall"] = tier_df["recall"].round(4)
tier_df["FTE_h"] = tier_df["FTE_h"].round(1)
tier_df["FTE_days"] = (tier_df["FTE_h"] / 8).round(2)
display(tier_df)

print(f"\nQuantile τ={q90:.4f} даёт {qm['alerts']} алертов на valid ({qm['alerts']/len(y_valid):.1%}) при precision {qm['precision']:.3f} — без учёта C_FN/C_FP.")
print(f"Tier 1 τ={tau1:.4f}: {tier_df.iloc[0]['alerts']} алертов, precision {tier_df.iloc[0]['precision']:.3f}, recall {tier_df.iloc[0]['recall']:.3f}, FTE {tier_df.iloc[0]['FTE_h']:.0f}ч ({tier_df.iloc[0]['FTE_days']:.1f} чел-дней)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

# precision / recall per tier
plot_tiers = tier_df.copy()
plot_tiers_m = plot_tiers.melt(id_vars="tier", value_vars=["precision", "recall"], var_name="metric", value_name="value")
sns.barplot(data=plot_tiers_m, x="tier", y="value", hue="metric", ax=axes[0])
axes[0].set_title("Precision / Recall по tier (valid)")
axes[0].set_xlabel("")
axes[0].set_ylabel("доля")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=12)
axes[0].legend(title="")
for c in axes[0].containers:
    axes[0].bar_label(c, fmt="%.2f", fontsize=7)

# alerts + FTE
sns.barplot(data=plot_tiers, x="tier", y="alerts", color="steelblue", ax=axes[1])
axes[1].set_title("Нагрузка: алерты и FTE (15 мин/алерт)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=12)
axes[1].axhline(B, color="red", linestyle=":", label=f"B={B}")
axes[1].legend()
for i, row in enumerate(plot_tiers.itertuples()):
    axes[1].text(i, row.alerts + 60, f"{row.alerts}\n{row.FTE_h:.0f}ч", ha="center", fontsize=7)

plt.tight_layout()
plt.show()

print("Вывод: cost-оптимальные Tier 1/2 доминируют квантиль по recall при заданной стоимости; Tier 3 — кандидат на auto-clear только после power-анализа.")


## 5. Power analysis для фазы auto-clear

Tier 3 (low-score) хотим автоматически закрывать без ручной проверки, но нужно доказать, что его precision (доля illicit среди "чистых") низка. Оцениваем $p$ — истинную precision Tier 3 — по случайной выборке. Для доли используем нормальную аппроксимацию:

$$n = \frac{z^2\,p(1-p)}{e^2}$$

где $z=1.96$ (95% CI), $e$ — допустимая ошибка. Пример: $p=0.05$, $e=0.01$ → $n\approx1825$. Это размер **случайной** выборки из Tier 3, которую нужно разметить вручную для несмещённой оценки. Без рандома (только "подозрительные" кейсы) оценка смещена.


In [ ]:
z = 1.96
p_example = 0.05
e = 0.01
n_example = (z**2 * p_example * (1 - p_example)) / (e**2)
print(f"n = z²·p(1-p)/e² = {z}²·{p_example}·{1-p_example}/{e}² = {n_example:.0f}")
print(f"→ нужно ~{int(np.ceil(n_example)):,} случайных транзакций Tier 3 для оценки precision ±{e:.0%} при p≈{p_example:.0%}")

# curve n vs p for several e
p_grid = np.linspace(0.01, 0.50, 100)
e_grid = [0.005, 0.01, 0.02]
curves = []
for ei in e_grid:
    n_vals = (z**2 * p_grid * (1 - p_grid)) / (ei**2)
    curves.append(pd.DataFrame({"p": p_grid, "n": n_vals, "e": f"e={ei:.3f}"}))
curve_df = pd.concat(curves, ignore_index=True)

fig, ax = plt.subplots(figsize=(7.5, 4.4))
sns.lineplot(data=curve_df, x="p", y="n", hue="e", ax=ax)
ax.axvline(p_example, color="grey", linestyle="--", alpha=0.7)
ax.axhline(n_example, color="grey", linestyle="--", alpha=0.7)
ax.scatter([p_example], [n_example], color="red", zorder=5)
ax.annotate(f"p={p_example:.2f}, e={e:.2f}\nn≈{n_example:.0f}", xy=(p_example, n_example), xytext=(p_example+0.08, n_example+600), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_xlabel("p — ожидаемая precision Tier 3 (доля illicit среди auto-clear)")
ax.set_ylabel("n — требуемый размер случайной выборки")
ax.set_title("Power analysis: n vs p для разных e (z=1.96, 95% CI)")
ax.legend(title="")
plt.tight_layout()
plt.show()

# also show n vs e for fixed p
e_vals = np.linspace(0.005, 0.05, 100)
n_vs_e = (z**2 * p_example * (1 - p_example)) / (e_vals**2)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(e_vals, n_vs_e)
ax.axvline(e, color="red", linestyle="--")
ax.set_xlabel("e — допустимая ошибка оценки precision")
ax.set_ylabel("n")
ax.set_title(f"n vs e при p={p_example:.2f}, z={z}")
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(72)

# unbiased estimation requires random sampling from Tier 3 population
# demo: Tier 3 predictions on valid
tier3_mask = valid_score < tau3  # low-score -> auto-clear candidate (invert of earlier tier def)
# Actually Tier 3 defined as score >=0.10 is alerts; auto-clear is score < tau3. Show both.
# For demo: auto-clear pool = score < 0.10
auto_clear_scores = valid_score[valid_score < tau3]
auto_clear_y = y_valid[valid_score < tau3]
print(f"Auto-clear пул (score < {tau3}): n={len(auto_clear_y):,}  истинная precision={auto_clear_y.mean():.4f} (доля illicit, должна быть ≈0)")

# simulate random sampling of n_example from this pool
n_sim = int(min(len(auto_clear_y), int(np.ceil(n_example))))
if n_sim >= 10:
    sample_idx = np.random.choice(len(auto_clear_y), size=n_sim, replace=False)
    p_hat = auto_clear_y[sample_idx].mean()
    se = np.sqrt(p_hat * (1 - p_hat) / n_sim) if 0 < p_hat < 1 else np.sqrt(0.25 / n_sim)
    ci_low, ci_high = p_hat - z * se, p_hat + z * se
    print(f"Случайная выборка n={n_sim}: p̂={p_hat:.4f}  95% CI [{max(0,ci_low):.4f}, {ci_high:.4f}]  SE={se:.4f}")
    print(f"Без рандома (берём топ-подозрительные из пула) оценка была бы смещена вверх — нельзя экстраполировать на весь Tier 3.")
else:
    print("Пул меньше требуемого n — нужно копить Tier 3 или увеличить e.")

# illustrate bias of selective sampling
fig, ax = plt.subplots(figsize=(7, 3.6))
# selective = highest scores within auto-clear pool (closest to threshold) -> inflated p
sorted_y = auto_clear_y[np.argsort(auto_clear_scores)]  # ascending score
k = min(200, len(sorted_y))
selective_p = sorted_y[-k:].mean() if k>0 else 0
vals = [p_hat if n_sim>=10 else 0, selective_p]
ax.bar(["random (unbiased)", f"top-{k} near τ (biased)"], vals, color=["steelblue", "tomato"])
ax.set_ylabel("оценка precision auto-clear")
ax.set_title("Почему нужна случайная выборка: селективная завышает риск")
# fix y-limit and label position for tiny values (0.0016)
max_v = max(vals) if vals else 0
ax.set_ylim(0, max(0.015, max_v * 1.6))
for i, v in enumerate(vals):
    ax.text(i, v + max(0.0004, max_v*0.04), f"{v:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## 6. Выводы

- **Cost-оптимум вместо квантиля.** $\tau^{\ast}=\arg\min Cost$ напрямую минимизирует бизнес-стоимость ($C_{FP}=1$, $C_{FN}=10/100$), тогда как квантиль (90-й перцентиль) калибруется по распределению скора и дрейфует между периодами (valid $\tau_{90}=q_{valid}$, test $\tau_{90}=q_{test}$). На valid $\tau^{\ast}(10)\approx$ выше, $\tau^{\ast}(100)\approx$ ниже — разница в 3× по алертам.
- **Бюджет спасает при высоком $C_{FN}$.** При $C_{FN}=100$ оптимум даёт >3000 алертов на valid (10 периодов) — больше $B=500$. Без ограничения $Alerts\le B$ очередь аналитиков захлебнётся; бюджет форсирует повышение $\tau$ до пересечения с $B$, даже ценой роста $FN$.
- **Tier-система операционализирует это.** Tier 1 ($\tau^{\ast}(100)$) — urgent, Tier 2 ($\tau^{\ast}(10)$) — review, Tier 3 ($\tau=0.10$) — bulk. FTE считается как $alerts\times0.25$ч; квантильный baseline при том же числе алертов проигрывает по recall/cost.
- **Power analysis даёт размер выборки для auto-clear.** Для проверки Tier 3 нужно $n=z^2p(1-p)/e^2$ случайных кейсов (≈1825 при $p=0.05$, $e=0.01$); только рандом даёт несмещённую оценку precision, селективная выборка завышает риск.
- **Ограничения.** RF — прокси GBDT без графовых признаков; valid-оценка оптимистична (PR-AUC 0.96 на valid vs 0.64 на test — дрифт). Пороги нужно перекалибровывать по скользящему valid и валидировать на test/out-of-time.
